## Import Headers

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
import json
from dotenv import load_dotenv
load_dotenv()

from helpers import extract_ordered_content

d:\code\ai-learning-projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Extract contents of PDF using PymuPDF (fitz) and pdfplumber:
- If Text  → keep as-is
- If Table → convert to markdown
- If Image → send to llm and get caption.

### 2.1 Extract Vendor Report PDF

In [2]:
# extract_ordered_content function definition is present in helpers.py

vendor_report = extract_ordered_content("data-reports/vendor_report.pdf")
print(f"Vendor Document: \n{json.dumps(vendor_report, indent=4)}")

Vendor Document: 
[
    {
        "type": "text",
        "content": "Water Test",
        "page": 1,
        "y": 76.73999786376953
    },
    {
        "type": "text",
        "content": "All tested components demonstrated vibration levels within acceptable US regulatory limits for water test.",
        "page": 1,
        "y": 126.8499755859375
    },
    {
        "type": "image",
        "content": "[IMAGE DESCRIPTION]: Bar chart titled \"4.1 Water Test Vibration Levels\" showing vibration measurements in mm/s for different part codes: L102 (2.23), M330 (3.77), X778 (2.57), A450 (2.37), B990 (2.99).",
        "page": 1,
        "y": 187.5999755859375
    },
    {
        "type": "text",
        "content": "Dust Test",
        "page": 2,
        "y": 76.73999786376953
    },
    {
        "type": "text",
        "content": "All tested components demonstrated vibration levels within acceptable US regulatory limits for dust test.",
        "page": 2,
        "y": 126.8499755859375
   

### 2.2 Extract US Regulations PDF

In [3]:
us_regulations = extract_ordered_content("data-reports/us_regulations.pdf")
print(f"US Regulations document: \n{json.dumps(us_regulations, indent=4)}")

US Regulations document: 
[
    {
        "type": "text",
        "content": "US Federal Automotive Vibration Compliance Standards",
        "page": 1,
        "y": 76.73999786376953
    },
    {
        "type": "text",
        "content": "This document defines per-component vibration compliance thresholds under different environmental exposure tests. Each part must satisfy its respective limit.",
        "page": 1,
        "y": 156.04998779296875
    },
    {
        "type": "table",
        "content": "[TABLE]:\n| Table 1.1 \u2013 Water Exposure Test Limits |  |\n| --- | --- |\n| Part Code | Vibration Limit (mm/s) |\n| L102 | 3.0 |\n| M330 | 4.5 |\n| X778 | 4.0 |\n| A450 | 3.0 |\n| B990 | 3.5 |\n",
        "page": 1,
        "y": 209.60000000000002
    },
    {
        "type": "table",
        "content": "[TABLE]:\n| Table 1.2 \u2013 Dust Exposure Test Limits |  |\n| --- | --- |\n| Part Code | Vibration Limit (mm/s) |\n| L102 | 5.0 |\n| M330 | 4.5 |\n| X778 | 2.5 |\n| A450 | 3.0 |\n|

## 3. Compliance Reasoning Prompt Template (Regulations vs Vendor Report)

In [4]:
# template = """
# You are a compliance analyst. First line of output - Overall complaint/ Non-Complaint followed Compliance for each tests.

# Regulatory thresholds:
# {us_regulations}

# Vendor measurements:
# {vendor_measurements}

# Question:
# {question}

# Instructions:
# - Compare each part carefully.
# - Identify violations.
# - Apply rejection rule as per mention in in regulations document.
# - Show reasoning clearly.
# """
template = """
You are a compliance analyst.

Return the answer strictly in the following format:

Overall: <Compliant / Non-Compliant>

Test-wise Breakdown:
For each test condition found in the documents:
- <Test Name>: <Compliant / Non-Compliant>

Explanation:
- Mention only the components that violate their limits (if any).
- Show the numeric comparison (measured vs threshold).
- Calculate the percentage of non-compliant components for that test.
- Apply the batch rejection rule exactly as stated in the regulations.
- Keep the explanation concise and structured.

Important:
- Do NOT repeat full regulatory tables.
- Do NOT repeat all measurements unless necessary.
- Focus only on violations and final reasoning.
- If no violations exist for a test, clearly state "No violations detected."

---

Regulatory thresholds:
{us_regulations}

Vendor measurements:
{vendor_measurements}

Question:
{question}
"""
prompt = PromptTemplate(
    input_variables=["us_regulations", "vendor_measurements", "question"],
    template=template
)

## 4. Run LLM Compliance Query

In [5]:
llm = ChatOpenAI(model="gpt-4o")

query = "Is the batch compliant under all tests?"

final_prompt = prompt.format(
    us_regulations=us_regulations,
    vendor_measurements=vendor_report,
    question=query
)

response = llm.invoke(final_prompt)

print(response.content)

Overall: Non-Compliant

Test-wise Breakdown:
- Water Test: Compliant
  - No violations detected.
- Dust Test: Compliant
  - No violations detected.
- Mud Test: Non-Compliant
  - X778: Measured 4.8 mm/s vs threshold 4.0 mm/s

Explanation:
- Mud Test: Part X778 exceeded the vibration limit (4.8 mm/s > 4.0 mm/s).
- Percentage of non-compliant components in Mud Test: 1 out of 5 (20% non-compliance).
- Batch Rejection Rule: Batch is rejected for Mud Test as non-compliance exceeds 10%.


> Note: Exactly one component measurement was intentionally set above the threshold in the mud test to validate the robustness of the pipeline.